# Bathymetry — GEBCO vs ETOPO1 resolution

Pull the **same** area from both shipped global DEMs — GEBCO 2020 at 15
arc-seconds and ETOPO1 at 1 arc-minute — and compare them side by side. The
Big Island of Hawaii gives a wide relief range (volcano summit above sea level
down to deep flanks), so the roughly 4x resolution difference is easy to see.

In [ ]:
import tempfile

import matplotlib.pyplot as plt
import numpy as np
from earthlens.core import EarthLens
from pyramids.dataset import Dataset

AOI = [-156.4, 18.6, -154.6, 20.4]  # [west, south, east, north]

In [ ]:
def fetch(dataset):
    out = tempfile.mkdtemp()
    paths = EarthLens(
        data_source='bathymetry', dataset=dataset, aoi=AOI, path=out
    ).download(progress_bar=False)
    grid = np.asarray(Dataset.read_file(str(paths[0])).read_array(), dtype='float32')
    grid = grid[0] if grid.ndim == 3 else grid
    return np.where(grid == 32767, np.nan, grid)


gebco = fetch('gebco_2020')
etopo = fetch('etopo1_ice')
print('GEBCO 15-arcsec grid:', gebco.shape)
print('ETOPO1 1-arcmin grid:', etopo.shape)

## Side by side

Same colour scale for a fair comparison; the finer GEBCO grid resolves rift
zones and submarine canyons that the coarser ETOPO1 grid blurs.

In [ ]:
vmin, vmax = -5000, 4000
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
panels = (
    (axes[0], gebco, f'GEBCO 2020 (15-arcsec) {gebco.shape}'),
    (axes[1], etopo, f'ETOPO1 (1-arcmin) {etopo.shape}'),
)
for ax, data, title in panels:
    im = ax.imshow(data, cmap='terrain', vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.axis('off')
fig.colorbar(im, ax=axes, label='elevation (m)', shrink=0.7)
plt.show()